In [13]:
from sklearn.datasets import fetch_california_housing

# 경고 무시
import warnings
warnings.filterwarnings('ignore')

dataset = fetch_california_housing() # 데이터셋을 불러옴
print(dataset.keys())   # 데이터셋의 키(요소들의 이름)를 출력

dict_keys(['data', 'target', 'frame', 'target_names', 'feature_names', 'DESCR'])


# 데이터의 구성요소 확인

In [14]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# ❶ 데이터셋 로드
dataset = fetch_california_housing()

# ❷ 데이터프레임 생성
dataFrame = pd.DataFrame(dataset["data"])
dataFrame.columns = dataset["feature_names"]

# ❸ 정답(target) 추가
dataFrame["target"] = dataset["target"]

# ➍ 확인
print(dataFrame.head())


   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  target  
0    -122.23   4.526  
1    -122.22   3.585  
2    -122.24   3.521  
3    -122.25   3.413  
4    -122.25   3.422  


# 선형회귀를 위한 MLP모델의 설계

In [32]:
import torch
import torch.nn as nn

from torch.optim.adam import Adam
from torch.optim import SGD


# 1. 모델 정의
model = nn.Sequential(
   nn.Linear(8, 100),
   nn.ReLU(),
   nn.Linear(100, 1)
)

# He initialization 적용
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight)
        nn.init.zeros_(m.bias)

X = dataFrame.iloc[:, :8].values # 2. 정답을 제외한 특징을 X에 입력
Y = dataFrame["target"].values # 데이터프레임의 target의 값을 추출

batch_size = 100
learning_rate = 0.001

# 3. 가중치를 수정하기 위한 최적화 정의
optim = Adam(model.parameters(), lr=learning_rate)
# optim = SGD(model.parameters(), lr=1e-4, momentum=0.9, nesterov=True)


# 에포크 반복
for epoch in range(200):

   # 배치 반복
   for i in range(len(X)//batch_size):
       start = i*batch_size      # 4. 배치 크기에 맞게 인덱스를 지정
       end = start + batch_size


       # 파이토치 실수형 텐서로 변환
       x = torch.FloatTensor(X[start:end])
       y = torch.FloatTensor(Y[start:end])

       optim.zero_grad() # 5. 가중치의 기울기를 0으로 초기화 -> 이전 그라디언트 지우기 위해
       preds = model(x)  # 6. 모델의 예측값 계산
       loss = nn.MSELoss()(preds, y) # 7. MSE 손실 계산
       loss.backward() # gradient 계산만
       optim.step() # 웨이트 업데이트 -> 웨이트 누적이 일어날수도 있으므로 바로 지우지 않음
   if epoch % 20 == 0:
       print(f"epoch{epoch} loss:{loss.item()}")

epoch0 loss:222.10629272460938
epoch20 loss:5.306541919708252
epoch40 loss:1.0942407846450806
epoch60 loss:1.0655927658081055
epoch80 loss:1.0202668905258179
epoch100 loss:1.145550012588501
epoch120 loss:1.1048239469528198
epoch140 loss:1.0283349752426147
epoch160 loss:1.0255987644195557
epoch180 loss:1.0544335842132568


# 모델 성능 평가

In [16]:
prediction = model(torch.FloatTensor(X[0, :8]))
real = Y[0]
print(f"prediction:{prediction.item()} real:{real}")

prediction:3.0251073837280273 real:4.526


# 최적화 모델 SGD 이용하기

In [34]:
import torch
import torch.nn as nn

from torch.optim.adam import Adam
from torch.optim import SGD


# 모델 정의
model = nn.Sequential(
   nn.Linear(8, 100),
   nn.ReLU(),
   nn.Linear(100, 1)
)

# He initialization 적용
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight)
        nn.init.zeros_(m.bias)

X = dataFrame.iloc[:, :8].values # 정답을 제외한 특징을 X에 입력
Y = dataFrame["target"].values # 데이터프레임의 target의 값을 추출

batch_size = 1
learning_rate = 0.001

# 가중치를 수정하기 위한 최적화 정의
# optim = Adam(model.parameters(), lr=learning_rate)
optim = SGD(model.parameters(), lr=1e-4, momentum=0.9, nesterov=True)


# 에포크 반복
for epoch in range(200):

   # 배치 반복
   for i in range(len(X)//batch_size):
       start = i*batch_size
       end = start + batch_size


       # 파이토치 실수형 텐서로 변환
       x = torch.FloatTensor(X[start:end])
       y = torch.FloatTensor(Y[start:end])

       optim.zero_grad()
       preds = model(x)
       loss = nn.MSELoss()(preds, y)
       loss.backward()
       optim.step()

   if epoch % 20 == 0:
       print(f"epoch{epoch} loss:{loss.item()}")

epoch0 loss:459.7337341308594
epoch20 loss:0.06240398436784744
epoch40 loss:0.04475846886634827
epoch60 loss:0.042539115995168686
epoch80 loss:0.042456887662410736
epoch100 loss:0.04405684396624565
epoch120 loss:0.042508091777563095
epoch140 loss:0.039501070976257324
epoch160 loss:0.0411684550344944
epoch180 loss:0.03988039866089821


In [ ]:
prediction = model(torch.FloatTensor(X[0, :8]))
real = Y[0]
print(f"prediction:{prediction.item()} real:{real}")

prediction:2.084846258163452 real:4.526


In [18]:
prediction = model(torch.FloatTensor(X[0, :8]))
real = Y[0]
print(f"prediction:{prediction.item()} real:{real}")

prediction:2.084846258163452 real:4.526


# initialization He(kaiming)으로 변경하기

In [30]:
import torch
import torch.nn as nn

from torch.optim.adam import Adam
from torch.optim import SGD


# ❶ 모델 정의
model = nn.Sequential(
   nn.Linear(8, 100),
   nn.ReLU(),
   nn.Linear(100, 1)
)

# He initialization 적용
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight)
        nn.init.zeros_(m.bias)

X = dataFrame.iloc[:, :8].values # ❷ 정답을 제외한 특징을 X에 입력
Y = dataFrame["target"].values    # 데이터프레임의 target의 값을 추출

batch_size = 100
learning_rate = 0.001

# ❸ 가중치를 수정하기 위한 최적화 정의
optim = Adam(model.parameters(), lr=learning_rate)


# 에포크 반복
for epoch in range(200):

   # 배치 반복
   for i in range(len(X)//batch_size):
       start = i*batch_size      # ➍ 배치 크기에 맞게 인덱스를 지정
       end = start + batch_size


       # 파이토치 실수형 텐서로 변환
       x = torch.FloatTensor(X[start:end])
       y = torch.FloatTensor(Y[start:end])

       optim.zero_grad() # ❺ 가중치의 기울기를 0으로 초기화
       preds = model(x)  # ❻ 모델의 예측값 계산
       loss = nn.MSELoss()(preds, y) # ❼ MSE 손실 계산
       loss.backward()
       optim.step()

   if epoch % 20 == 0:
       print(f"epoch{epoch} loss:{loss.item()}")

epoch0 loss:63.29792022705078
epoch20 loss:23.76543426513672
epoch40 loss:2.095548629760742
epoch60 loss:1.9777017831802368
epoch80 loss:1.74813711643219
epoch100 loss:1.7599852085113525
epoch120 loss:1.733160138130188
epoch140 loss:1.6335104703903198
epoch160 loss:1.5329415798187256
epoch180 loss:1.545872688293457


In [31]:
prediction = model(torch.FloatTensor(X[0, :8]))
real = Y[0]
print(f"prediction:{prediction.item()} real:{real}")

prediction:1.3195514678955078 real:4.526


# MAE + SGD 적용하기
### loss는 가장 적은데 predict는 너무 이상함

In [25]:
import torch
import torch.nn as nn

from torch.optim.adam import Adam
from torch.optim import SGD


# ❶ 모델 정의
model = nn.Sequential(
   nn.Linear(8, 100),
   nn.ReLU(),
   nn.Linear(100, 1)
)

# He initialization 적용
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight)
        nn.init.zeros_(m.bias)

X = dataFrame.iloc[:, :8].values # ❷ 정답을 제외한 특징을 X에 입력
Y = dataFrame["target"].values    # 데이터프레임의 target의 값을 추출

batch_size = 100
learning_rate = 0.001

# ❸ 가중치를 수정하기 위한 최적화 정의
optim = Adam(model.parameters(), lr=learning_rate)
optim = SGD(model.parameters(), lr=1e-4, momentum=0.9, nesterov=True)


# 에포크 반복
for epoch in range(200):

   # 배치 반복
   for i in range(len(X)//batch_size):
       start = i*batch_size      # ➍ 배치 크기에 맞게 인덱스를 지정
       end = start + batch_size


       # 파이토치 실수형 텐서로 변환
       x = torch.FloatTensor(X[start:end])
       y = torch.FloatTensor(Y[start:end])

       optim.zero_grad() # ❺ 가중치의 기울기를 0으로 초기화 -> 이전 그라디언트 지우기 위해
       preds = model(x)  # ❻ 모델의 예측값 계산
       loss = nn.L1Loss()(preds, y) # ❼ MAE 손실 계산
       loss.backward()
       optim.step()

   if epoch % 20 == 0:
       print(f"epoch{epoch} loss:{loss.item()}")

epoch0 loss:1.8254822492599487
epoch20 loss:0.6707932353019714
epoch40 loss:0.7260346412658691
epoch60 loss:0.7462395429611206
epoch80 loss:0.7485753893852234
epoch100 loss:0.7490094900131226
epoch120 loss:0.7498917579650879
epoch140 loss:0.7496312856674194
epoch160 loss:0.7472904920578003
epoch180 loss:0.7457575798034668


In [26]:
prediction = model(torch.FloatTensor(X[0, :8]))
real = Y[0]
print(f"prediction:{prediction.item()} real:{real}")

prediction:1.8360031843185425 real:4.526
